In [1]:
#This notebook will serve as a baseline with the regular
#Single Agent logic implemented in gymnasium as was done in 
#MATLAB
#This will allow us to try certain concepts in a controlled
#environment rather than the more complex MARL environment


In [22]:
import gymnasium as gym
import numpy as np
import matplotlib
from typing import Optional
from gymnasium import spaces
import random

In [23]:
#Setup environment
class SingleSatelliteEnv(gym.Env):

    def __init__(self):
        #We define the key environment variables here
        
        
        
        #Observation space for the satellite
        #Contact plan
        #Current data to downlink
        self.timestep=None
        self.contact_plan=np.full((10,), None)
        self.satellite_remaining_data=None
        self.delivered_data=None
        self.Energy_Expended=None
        self.initial_data_volume=None
        self.num_of_contacts=None
        self.observation_space = gym.spaces.Box(
    low=np.array([0.0]*10 + [0.0] + [0.0], dtype=np.float32),
    high=np.array([1.0]*10 + [1.0] + [10.0], dtype=np.float32),
    dtype=np.float32
)
        
        self.action_space = gym.spaces.Discrete(2)
    
    
    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Start a new episode.
        
        Args:
            seed: Random seed for reproducible episodes
            options: Additional configuration (unused in this example)
        
        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)
        self.timestep=0
        self.contact_plan=self.np_random.uniform(low=0.0, high=1.0, size=(10,)).astype(np.float32)
        self.contact_plan= np.round(self.contact_plan*10)/10
        self.satellite_remaining_data=np.array(
        self.np_random.integers(5, 100) / 100.0, dtype=np.float32
    )
        self.delivered_data=0
        self.initial_data_volume=self.satellite_remaining_data
        self.Energy_Expended=0
        self.num_of_contacts=0
        observation = self._get_obs()
        info = self._get_info()
        return observation, info
    def updateDeliveryandEnergy(self,weather,length,remaining_data):
        delivered_packets=0
        excess_energy_expended=0
        random_sample=self.np_random.uniform(low=0.0, high=1.0, size=(10,)).astype(np.float32)
        for i in range(length-1):
            if remaining_data>0:
                if random_sample[i]> weather:
                    delivered_packets=delivered_packets+1
                    remaining_data=remaining_data-1
                else:
                    excess_energy_expended=excess_energy_expended+1
            else:
                excess_energy_expended=excess_energy_expended+1
        
        return delivered_packets,excess_energy_expended
    def step(self, action):
        if action==0:
            reward=0
        else:
            self.num_of_contacts+=1
            current_delivered_data,current_energy_expenditure=self.updateDeliveryandEnergy(self.contact_plan[self.timestep],10,self.satellite_remaining_data*100)
            if current_delivered_data>0:
                reward=(10/(self.initial_data_volume*100))*current_delivered_data-5/(100*self.initial_data_volume)*current_delivered_data*(current_energy_expenditure/(current_energy_expenditure+10))
            else:
                reward=-(5*current_energy_expenditure)/(self.initial_data_volume*100)
            self.delivered_data=self.delivered_data+current_delivered_data
            self.Energy_Expended=self.Energy_Expended+current_energy_expenditure
            self.satellite_remaining_data=(self.satellite_remaining_data)*100-current_delivered_data
            self.satellite_remaining_data=self.satellite_remaining_data/100
        self.updateTimestep()
        terminated=False
        truncated=False
        if self.timestep>=10 and self.satellite_remaining_data>0:
            terminated=True
            reward=reward+10*(self.delivered_data/(self.initial_data_volume*100))-5*(self.Energy_Expended/100)
        elif self.satellite_remaining_data<=0:
            truncated=True
            #Episode Reward
            reward=reward+10-(self.Energy_Expended/(10*(self.timestep)))*5
        observation = self._get_obs()
        info = self._get_info()
        reward=float(reward)
        return observation, reward, terminated, truncated, info
    def _get_info(self):
      
        return {"contact plan":self.contact_plan, "Remaining data":self.satellite_remaining_data,"current timestep":self.timestep,"Energy Expended":self.Energy_Expended,"Delivered Data":self.delivered_data,"Initial Data":self.initial_data_volume,"Number of Contacts":self.num_of_contacts}
    def _get_obs(self):
       return np.concatenate([
        np.array(self.contact_plan, dtype=np.float32),
        np.array([self.satellite_remaining_data], dtype=np.float32),
        np.array([self.timestep], dtype=np.float32)
    ])
        #return {"contact plan": self.contact_plan, "Remaining data": self.satellite_remaining_data,"current timestep":np.array([float(self.timestep)], dtype=np.float32)}
    def updateTimestep(self):
        self.timestep=self.timestep+1
        return self.timestep
    def updateDeliveredPackets(self,new_deliveries):
        self.delivered_data=self.delivered_data+new_deliveries
    def getDeliveredPackets(self):
        return self.delivered_data
    def getEnergyExpenditure(self):
        return self.Energy_Expended
    def updateEnergyExpenditure(self,new_expend):
        self.Energy_Expended=self.Energy_Expended+new_expend
    def updateRemainingData(self,new_deliveries):
        self.satellite_remaining_data=(self.satellite_remaining_data)*100-current_delivered_data
        self.satellite_remaining_data=self.satellite_remaining_data/100
        

In [25]:
#Next need to produce training graphs and compare between pre-existing agents
from gymnasium.envs.registration import register
register(
    id="SingleSatelliteEnv",
    entry_point="Single_ag_Environment:SingleSatelliteEnv",  # Adjust the module path
)

# Test if the environment works
#env = gym.make("SingleSatelliteEnv")
#obs, info = env.reset()
#print("Custom Env Loaded Successfully!")

obs: [0.2  0.   0.6  0.8  0.4  0.2  0.3  1.   0.8  0.3  0.78 0.  ]
dtype: float32
shape: (12,)
Custom Env Loaded Successfully!
